In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

from hallucinations_kg.utils.logging import set_error_level

sys.path.append("..")


import json
from functools import partial
from typing import Any

import pandas as pd
from datasets import disable_caching
from hydra import compose, initialize
from pyvis.network import Network
from sklearn.metrics import precision_recall_curve

from hallucinations_kg.data.postprocessing import (
    add_allowed_nodes_and_relationships,
    parse_graph,
    postprocess_ents_rels,
)
from hallucinations_kg.data.utils import get_processed_dataset
from hallucinations_kg.metrics import auc_pr
from hallucinations_kg.models.baselines.selfcheckgpt import SelfCheckGPTPredictor
from hallucinations_kg.models.predictor import FactOccurrencePredictor, RandomFactPredictor
from hallucinations_kg.models.prompt_predictor import FactPromptPredictor, FactTextPromptPredictor
from hallucinations_kg.prediction.prediction import (
    add_fact_predictor_results,
)

disable_caching()
set_error_level()

## Loading

In [3]:
with initialize(version_base="1.3", config_path="config"):
    cfg = compose(config_name="fact_level_eval")
dataset = get_processed_dataset(cfg.processed_dataset)

Map:   0%|          | 0/238 [00:00<?, ? examples/s]

In [4]:
restrictions_column = cfg.processed_dataset.original_dataset.response_column
response_sentences_column = cfg.processed_dataset.original_dataset.response_sentences_column
samples_column = cfg.processed_dataset.original_dataset.samples_column

dataset = dataset.map(
    partial(
        add_allowed_nodes_and_relationships,
        restrictions_column=restrictions_column,
        response_sentences_column=response_sentences_column,
    )
)
dataset = dataset.map(
    partial(
        parse_graph,
        samples_column=samples_column,
        sentences_column=response_sentences_column,
        restrict=False,
        postprocess=False,
    )
)
dataset = dataset.rename_column("gpt3_sentences_graph", "gpt3_sentences_graph_no_postprocessing")
dataset = dataset.map(
    partial(
        parse_graph,
        samples_column=samples_column,
        sentences_column=response_sentences_column,
        restrict=True,
    )
)
dataset = dataset.map(partial(postprocess_ents_rels, response_column=restrictions_column))

Map:   0%|          | 0/238 [00:00<?, ? examples/s]

Map:   0%|          | 0/238 [00:00<?, ? examples/s]

Map:   0%|          | 0/238 [00:00<?, ? examples/s]

Map:   0%|          | 0/238 [00:00<?, ? examples/s]

In [5]:
import re


def parse_annotation(answer: str) -> int:
    answer = re.findall(r"[\w]+|[.,!?;\"']", answer)
    if "yes" in answer:
        return 0
    elif "no" in answer:
        return 1
    else:
        raise


def parse_fact_level_annotations(entry):
    annotations = []
    for sentence_annotation in entry["gpt_sentences_fact_level_annotations"]:
        annotations.append([parse_annotation(a) for a in sentence_annotation])
    entry["gpt_sentences_fact_level_annotations"] = annotations
    return entry


dataset = dataset.map(parse_fact_level_annotations)

Map:   0%|          | 0/238 [00:00<?, ? examples/s]

In [6]:
PREDICTOR_AGGREGATION_FUNCTIONS = [
    FactOccurrencePredictor,
    RandomFactPredictor,
    FactPromptPredictor,
    FactTextPromptPredictor,
    SelfCheckGPTPredictor,
]

for name, ds in dataset.items():
    ds = ds.map(
        partial(
            add_fact_predictor_results,
            predictor_aggregation_functions=PREDICTOR_AGGREGATION_FUNCTIONS,
        )
    )
    dataset[name] = ds

Map:   0%|          | 0/238 [00:00<?, ? examples/s]

In [7]:
df = dataset["evaluation"].to_pandas()
FACT_SCORES_COLUMNS = [col for col in df.columns if col.startswith("fact_score_")]
COLUMNS_TO_EXPLODE = [
    "gpt3_sentences",
    "gpt3_sentences_graph",
    "gpt3_sentences_graph_no_postprocessing",
    "gpt_sentences_fact_level_annotations",
    "binary_annotation",
] + FACT_SCORES_COLUMNS

sentence_df = df.explode(COLUMNS_TO_EXPLODE)
fact_df = sentence_df.explode(
    ["gpt3_sentences_graph", "gpt_sentences_fact_level_annotations"] + FACT_SCORES_COLUMNS
)
fact_df = fact_df[fact_df.gpt_sentences_fact_level_annotations.notna()]
fact_df["gpt3_sentences_graph"] = fact_df["gpt3_sentences_graph"].apply(tuple)
fact_df = fact_df.reset_index()
fact_df = fact_df.rename(columns={"index": "doc_id"})

In [8]:
print("Size of the dataset: ", len(df))
print("Size of sentence_df: ", len(sentence_df))

Size of the dataset:  238
Size of sentence_df:  1908


In [9]:
fact_df = fact_df.drop_duplicates(subset=["doc_id", "gpt3_sentences_graph"])
fact_df[fact_df.duplicated(subset=["doc_id", "gpt3_sentences_graph"])]
[selfcheckgpt_col] = [col for col in fact_df.columns if "SelfCheckGPT" in col]
avg_selfcheckgpt = fact_df.groupby(["doc_id", "gpt3_sentences_graph"])[selfcheckgpt_col].mean()

In [10]:
print("Size of fact_df: ", len(fact_df))

Size of fact_df:  5488


In [11]:
fact_df[selfcheckgpt_col] = fact_df.apply(
    lambda x: avg_selfcheckgpt.loc[(x["doc_id"], x["gpt3_sentences_graph"])], axis=1
)

In [12]:
def parse_params(params: str) -> dict[str, Any]:
    return json.loads(params.replace("'", '"'))


results = []
for col in FACT_SCORES_COLUMNS:
    y_pred = fact_df[col]
    y_true = fact_df.gpt_sentences_fact_level_annotations.to_numpy(dtype=float)
    y_pred = [x if pd.notna(x) else 0.5 for x in y_pred]

    auc_pr_hallucination = auc_pr(y_true, y_pred) * 100
    auc_pr_factual = auc_pr([1 - x for x in y_true], [1 - x for x in y_pred]) * 100
    prec, recall, thresholds = precision_recall_curve(y_true, y_pred)
    results.append(
        {
            "col": col,
            **parse_params(col.split("__")[-1]),
            "auc_factual": auc_pr_factual,
            "auc_hallucination": auc_pr_hallucination,
            "Precision": prec,
            "Recall": recall,
            "Thresholds": thresholds,
        }
    )

results = pd.DataFrame(results)

In [13]:
def get_method_name(row: pd.Series) -> str:
    if row["sentence_predictor"] == "SelfCheckGPT-reproduced":
        return "SelfCheckGPT-reproduced"
    else:
        if row["fact_predictor"] == "FactOccurrence":
            fact_predictor = "FactKGOccurrence"
        elif row["fact_predictor"] == "FactPrompt":
            fact_predictor = "FactKGPrompt"
        elif row["fact_predictor"] == "FactTextPrompt":
            fact_predictor = "FactTextPrompt"
        else:
            fact_predictor = row["fact_predictor"]
        return fact_predictor


results["Method"] = results.apply(get_method_name, axis=1)
to_report = results[["Method", "auc_hallucination"]]

to_report = to_report.rename(
    columns={
        "auc_hallucination": "AUC-PR-Hallucination",
        "auc_factual": "AUC-PR-Factual",
        "agg_func": "Agg",
    }
)

to_report = to_report.sort_values(by="AUC-PR-Hallucination", ascending=False)
to_report = to_report.rename(columns={"AUC-PR-Hallucination": "AUC-PR"})
display(to_report)

print(to_report.to_latex(index=False, float_format="%.2f"))

,Method,AUC-PR
3,FactSelfCheck-Text,93.414272
2,FactSelfCheck-KG (LLM-based),92.254878
0,FactSelfCheck-KG (Frequency-based),87.994074
4,SelfCheckGPT-reproduced,86.178808
1,RandomFact,65.794309


\begin{tabular}{lr}
\toprule
Method & AUC-PR \\
\midrule
FactSelfCheck-Text & 93.41 \\
FactSelfCheck-KG (LLM-based) & 92.25 \\
FactSelfCheck-KG (Frequency-based) & 87.99 \\
SelfCheckGPT-reproduced & 86.18 \\
RandomFact & 65.79 \\
\bottomrule
\end{tabular}



In [14]:
fact_df = sentence_df.explode(
    [
        "gpt3_sentences_graph",
        "gpt3_sentences_graph_no_postprocessing",
        "gpt_sentences_fact_level_annotations",
    ]
    + FACT_SCORES_COLUMNS
)
fact_df = fact_df[fact_df.gpt_sentences_fact_level_annotations.notna()]
fact_df["gpt3_sentences_graph"] = fact_df["gpt3_sentences_graph"].apply(tuple)
fact_df["gpt3_sentences_graph_no_postprocessing"] = fact_df[
    "gpt3_sentences_graph_no_postprocessing"
].apply(tuple)
fact_df = fact_df.reset_index()
fact_df = fact_df.rename(columns={"index": "doc_id"})

In [15]:
[factselfcheck_col] = [col for col in fact_df.columns if "FactSelfCheck-Text" in col]

In [16]:
fact_df["score_diff"] = fact_df[factselfcheck_col] - fact_df[selfcheckgpt_col]

In [17]:
p, r, t = results[results.col == factselfcheck_col][["Precision", "Recall", "Thresholds"]].iloc[0]
f1 = 2 * p * r / (p + r)
best_t = t[f1.argmax()]
print(f"FactSelfCheck best threshold: {best_t}")

FactSelfCheck best threshold: 0.4


In [18]:
doc_ids = [172]
for doc_id in doc_ids:
    doc_df = fact_df[fact_df.doc_id == doc_id]
    wiki_bio_text = doc_df["wiki_bio_text"].values[0]
    print(wiki_bio_text)
    for sentence in doc_df["gpt3_sentences"].unique():
        print(sentence)
        df_ = doc_df[doc_df["gpt3_sentences"] == sentence]
        print(
            df_[["gpt3_sentences_graph_no_postprocessing", factselfcheck_col]].to_latex(
                index=False, float_format="%.2f"
            )
        )
    print("\n\n")

    triples = doc_df["gpt3_sentences_graph_no_postprocessing"].values
    values = doc_df[factselfcheck_col]
    g = Network(notebook=True, directed=True)
    g.set_options("""
	var options = {
		"nodes": {
			"size":13,
			"font": {
				"size": 11
			}
		},
		"edges": {
			"arrowStrikethrough": false,
			"color": {
			"inherit": true
			},
			"font": {
			"size": 8,
			"align": "top"
			},
			"smooth": false
		},
		"manipulation": {
			"enabled": true,
			"initiallyActive": true
		},
		"physics": {
			"barnesHut": {
				"centralGravity": 0.2,
				"springLength": 100,
				"springConstant": 0.01,
				"damping": 0.7,
				"avoidOverlap": 1
			},
		"maxVelocity": 5,
		"minVelocity": 0.47,
		"solver": "barnesHut"
		}
	}
	""")
    for node in doc_df["allowed_nodes"].iloc[0]:
        g.add_node(node)
    for triple, value in zip(triples, values):
        color = "red" if value >= best_t else "green"
        g.add_edge(triple[0], triple[2], label=triple[1], width=(0.3 + value) * 4, color=color)
    g.show("nx.html")

Kenan Hasagić (born 1 February 1980) is a Bosnian football goalkeeper who plays for Balıkesirspor. His football career began in his hometown with FK Rudar. At the age of 16, he made his debut in a first division match. He was the most promising goalkeeper in Bosnia and Herzegovina; he played for youth selections and was later transferred to Austrian side Vorwärts Steyr. After that, he was a member of Altay SK in Turkey but didn't see much first team football. He went back to Bosnia and played for Bosna Visoko. In 2003, he signed a contract with FK Željezničar. Here he found good form and even became first choice goalkeeper for the Bosnian national team. In the 2004–05 season, he moved to Turkey once again where he signed for Turkish Süper Lig side Gaziantepspor. He made his debut for the national team on 12 February 2003 in a game between Wales and Bosnia and Herzegovina which ended in a 2–2 draw.
Kenan Hasagić (born 28 April 1988) is a Bosnian professional footballer who plays as a mi